# Workshop: Gemma from Scratch
## Notebook 2: Causal Masking

**Estimated Time: 10 minutes**

In a decoder-only model like Gemma, when we predict the next token, we must ensure the model only uses information from the current and previous tokens. If the model can "see" the answer during training, it won't learn anything!

### Learning Objectives:
1. Understand the concept of Causal Masking.
2. Implement a triangular mask using `torch.triu`.
3. Learn about Gemma's Sliding Window Attention mask.

In [ ]:
import torch
import matplotlib.pyplot as plt
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

seq_len = 10
sliding_window = 3

### 1. The Global Causal Mask

We want a matrix where position $(i, j)$ is `True` (or 1) if token $i$ is NOT allowed to see token $j$.
For causality, token $i$ can only see $j$ if $j \le i$.

We use `torch.triu(..., diagonal=1)` to get the upper triangle (excluding the main diagonal).

In [ ]:
ones = torch.ones((seq_len, seq_len), dtype=torch.bool)
mask_global = torch.triu(ones, diagonal=1)

plt.imshow(mask_global)
plt.title("Global Causal Mask (Purple=Visible, Yellow=Masked)")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.show()

### 2. Sliding Window Mask

Gemma often uses Sliding Window Attention (SWA). This means a token only attends to a fixed number of previous tokens (the "window"). This reduces memory usage for long sequences.

We need to mask out tokens that are too far in the past.

In [ ]:
# Mask for tokens too far in the past
far_past = torch.triu(ones, diagonal=sliding_window).T

# Combine with global causal mask
mask_local = mask_global | far_past

plt.imshow(mask_local)
plt.title(f"Sliding Window Mask (window={sliding_window})")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.show()

### 3. Applying the Mask

In the attention mechanism, we apply this mask to the scores *before* the softmax by setting masked positions to $-\infty$.

In [ ]:
scores = torch.randn(seq_len, seq_len)
masked_scores = scores.masked_fill(mask_global, float('-inf'))

print("Raw Scores (top-left 3x3):\n", scores[:3, :3])
print("\nMasked Scores (top-left 3x3):\n", masked_scores[:3, :3])

weights = torch.softmax(masked_scores, dim=-1)
print("\nWeights (Note the zeros in the upper triangle):\n", weights[:3, :3])

### Exercise:
Create a function `get_sliding_window_mask(seq_len, window_size)` that returns the combined mask.

**Hints:**
1. Create a boolean matrix of ones.
2. Use `torch.triu(..., diagonal=1)` for the causal part.
3. Use `torch.triu(..., diagonal=window_size).T` for the sliding window part (tokens too far back).
4. Combine them using the OR operator `|`.

In [ ]:
def get_sliding_window_mask(seq_len, window_size):
    # Your code here
    pass

# Test your function
try:
    test_mask = get_sliding_window_mask(seq_len, sliding_window)
    if test_mask is not None:
        assert torch.equal(test_mask, mask_local)
        print("✅ Success! Your mask matches the expected result.")
    else:
        print("Implement the function to test it.")
except Exception as e:
    print(f"❌ Error: {e}")

<details>
<summary><b>Click to see solution</b></summary>

```python
def get_sliding_window_mask(seq_len, window_size):
    ones = torch.ones((seq_len, seq_len), dtype=torch.bool)
    mask_global = torch.triu(ones, diagonal=1)
    far_past = torch.triu(ones, diagonal=window_size).T
    return mask_global | far_past
```
</details>